In [1]:
import pandas as pd
import sqlite3

# ---------- EXTRACT ----------
raw_data = {
    'OrderID':   [1, 2, 3, 4],
    'ProductID': [101, 102, 101, 103],
    'UnitPrice': [100, 200, 100, 300],
    'Quantity':  [2, 1, 3, 2],
    'OrderDate': ['2025-01-01', '2025-01-02', '2025-01-02', '2025-01-03'],
    'Gender':    ['Male', 'Female', 'Female', 'Male']
}
df_raw = pd.DataFrame(raw_data)
df_raw.to_csv('raw_orders.csv', index=False)
extracted_df = pd.read_csv('raw_orders.csv')
print("Extracted:\n", extracted_df)

# ---------- TRANSFORM ----------
df = extracted_df.copy()
df['OrderDate'] = pd.to_datetime(df['OrderDate'])
df['GenderCode'] = df['Gender'].map({'Male': 'M', 'Female': 'F'})
df['LineTotal'] = df['UnitPrice'] * df['Quantity']
df.drop(columns=['Gender'], inplace=True)
print("\nTransformed:\n", df)

# ---------- LOAD ----------
conn = sqlite3.connect('sales.db')
df.to_sql('Orders', conn, if_exists='replace', index=False)
print("\nData loaded into sales.db -> Orders table")

# ---------- VERIFY ----------
df_db = pd.read_sql("SELECT * FROM Orders", conn)
print("\nFrom database:\n", df_db)
conn.close()
print("\nDone.")

Extracted:
    OrderID  ProductID  UnitPrice  Quantity   OrderDate  Gender
0        1        101        100         2  2025-01-01    Male
1        2        102        200         1  2025-01-02  Female
2        3        101        100         3  2025-01-02  Female
3        4        103        300         2  2025-01-03    Male

Transformed:
    OrderID  ProductID  UnitPrice  Quantity  OrderDate GenderCode  LineTotal
0        1        101        100         2 2025-01-01          M        200
1        2        102        200         1 2025-01-02          F        200
2        3        101        100         3 2025-01-02          F        300
3        4        103        300         2 2025-01-03          M        600

Data loaded into sales.db -> Orders table

From database:
    OrderID  ProductID  UnitPrice  Quantity            OrderDate GenderCode  \
0        1        101        100         2  2025-01-01 00:00:00          M   
1        2        102        200         1  2025-01-02 00:00:0